## **Introduction to Middleware**

### **What is Middleware?**
Middleware is a feature in LangChain that lets you step inside an agent’s workflow and make changes before, during, or after the model does something.

Think of it like a checkpoint system where you can plug in extra logic — without modifying the main agent code.

### **What You Can Do With Middleware?**
Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

### **How Middleware Work?**
The core agent loop involves:
1. calling a model,
2. letting it choose tools to execute, and
3. then finishing when it calls no more tools

Middleware exposes hooks before and after each of those steps.
<div style="display:flex; gap:20px;">
    <img src="assets/agent_loop.png" style="width:40%; height:auto;">
    <img src="assets/agent_loop_with_middlewares.png" style="width:40%; height:auto;">
</div>


## **PII Middleware**
Detect and handle Personally Identifiable Information (PII) in conversations. This middleware detects common PII types and applies configurable strategies to handle them. It can detect emails, credit cards, IP addresses, MAC addresses, and URLs in both user input and agent output.

**Configuration options:**
- pii_type: `Literal['email', 'credit_card', 'ip', 'mac_address', 'url']` | `str`
- strategy:
    - block: Raise an exception when PII is detected
    - redact: Replace PII with `[REDACTED_TYPE]` placeholders
    - mask: Partially mask PII (e.g., `****-****-****-1234` for credit card)
    - hash: Replace PII with deterministic hash (e.g., `<email_hash:a1b2c3d4>`)
- apply_to_input: bool = True
- apply_to_output: bool = False
- apply_to_tool_results: bool = False

In [11]:
from langchain_groq import ChatGroq

# Setup API Key
f = open('keys/.groq_api_key.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="llama-3.1-8b-instant", 
    temperature=1
)

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model=chat_model,
    tools=[],
    middleware=[
        PIIMiddleware(pii_type="email", strategy="redact", apply_to_input=True, apply_to_output=True),
        PIIMiddleware(pii_type="credit_card", strategy="mask", apply_to_input=True, apply_to_output=True),
    ],
)

response = agent.invoke(
    {
        "messages": "generate a markdown table with 5 random datapoints with features like name, email, ip addresses and credit card number"
    }
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

generate a markdown table with 5 random datapoints with features like name, email, ip addresses and credit card number
================================== Ai Message ==================================

I can generate a Markdown table with 5 random datapoints, but keep in mind that generating real credit card numbers is difficult due to the strict regulations around their use and format. I will use a placeholder in the format "XXXX-XXXX-XXXX-XXXX" for the credit card number, as actual credit card data is sensitive and should never be shared lightly.

### Random Datapoints Table
| Name        | Email                | IP Address       | Credit Card   | Country  |
|-------------|----------------------|------------------|----------------|----------|
| Emily Chen  | [REDACTED_EMAIL]    | 192.168.1.100    | 1234-5678-9012-3456  | USA      |
| Ethan Lee   | [REDACTED_EMAIL]     | 10.0.0.1         | 5678-9012-3456-

## **Summarization Middleware**

The summarization middleware monitors message token counts and automatically summarizes older messages when thresholds are reached.

**Configuration options:**
- model: The language model to use for generating summaries.
- summary_prompt: Prompt template for generating summaries. There exist a default prompt template.
- trigger: One or more thresholds that trigger summarization.
    - `("messages", 50)`: Trigger summarization when 50 messages is reached
    - `("tokens", 3000)`: Trigger summarization when 3000 tokens is reached
    - `[("fraction", 0.8), ("messages", 100)]`: Trigger summarization either when 80% of model's max input tokens is reached or when 100 messages is reached (whichever comes first)
- keep: Context retention policy applied after summarization.
    - `("messages", 20)`: Keep the most recent 20 messages
    - `("tokens", 3000)`: Keep the most recent 3000 tokens
    - `("fraction", 0.3)`: Keep the most recent 30% of the model's max input tokens
- trim_tokens_to_summarize: Maximum tokens to keep when preparing messages for the summarization call. Pass `None` to skip trimming entirely. Default to `4000` tokens.

In [3]:
# ! pip install wikipedia langchain-community

In [4]:
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(
    top_k_results=1,
    doc_content_chars_max=20_000,
)

In [5]:
from langchain_core.tools import tool

@tool
def fetch_wikipedia_data(query: str) -> str:
    """Fetch content of Wikipedia page from top hit of a query."""
    results = retriever.invoke(query)
    if results:
        return results[0].page_content
    return "(No data found)"

In [8]:
from langchain_groq import ChatGroq

# Setup API Key
f = open('keys/.groq_api_key.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="openai/gpt-oss-20b", 
    temperature=1
)

summarization_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="llama-3.1-8b-instant", 
    temperature=1
)

In [6]:
summary_prompt = """
Summarize the main thrust of this conversation. What have the human and assistant
discussed so far? Focus on key facts and requests.
<messages>
Messages to summarize:
{messages}
</messages>
"""

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=chat_model,
    tools=[fetch_wikipedia_data],
    middleware=[
        SummarizationMiddleware(
            model=summarization_model,
            summary_prompt=summary_prompt,
            # Trigger summarization when 70% of context is used
            trigger=("fraction", 0.7),
            # Keep the most recent 30% of messages in full
            keep=("fraction", 0.3),
            # No additional trimming before summarization
            trim_tokens_to_summarize=None,
        ),
    ],
)

## **Model Call Limit Middleware**

Limit the number of model calls to prevent infinite loops or excessive costs. 

Model call limit is useful for the following:
- Preventing runaway agents from making too many API calls.
- Enforcing cost controls on production deployments.
- Testing agent behavior within specific call budgets.

This middleware monitors the number of model calls made during agent execution and can terminate the agent when specified limits are reached. It supports both **thread-level** and **run-level** call counting with configurable exit behaviors.
- **Thread-level (i.e. Conversation Level)**: The middleware tracks the number of model calls and persists call count across multiple runs (invocations) of the agent.
- **Run-level (i.e. Single Invocation)**: The middleware tracks the number of model calls made during a single run (invocation) of the agent.

**Configuration options:**
- thread_limit: Maximum model calls across all runs in a thread. Defaults to no limit. For thread_limit, it is mandatory to provide a `checkpointer`
- run_limit: Maximum model calls per single invocation. Defaults to no limit.
- exit_behavior: Behavior when limit is reached. Options: `end` (graceful termination) or `error` (raise exception). Default to `end`.

In [5]:
from langchain_core.tools import tool

@tool
def multiply_tool(a: int, b: int) -> int:
    """This tool take two integer variables in the input and returns the product"""
    return a * b

In [8]:
from langchain_groq import ChatGroq

# Setup API Key
f = open('keys/.groq_api_key.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="openai/gpt-oss-20b", 
    temperature=1
)


In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware

# Create middleware with limits
call_tracker = ModelCallLimitMiddleware(run_limit=2, exit_behavior="end")

agent = create_agent(
    model=chat_model,
    tools=[multiply_tool],
    middleware=[call_tracker]
)

In [10]:
response = agent.invoke({"messages": "help me multiply 3 and 4"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

help me multiply 3 and 4
================================== Ai Message ==================================
Tool Calls:
  multiply_tool (fc_3198d2be-500e-4505-9e78-fc8dec9c12f7)
 Call ID: fc_3198d2be-500e-4505-9e78-fc8dec9c12f7
  Args:
    a: 3
    b: 4
================================= Tool Message =================================
Name: multiply_tool

12
================================== Ai Message ==================================

The product of 3 and 4 is **12**.


In [11]:
response = agent.invoke({"messages": "help me multiply 3 and 4 and 5"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

help me multiply 3 and 4 and 5
================================== Ai Message ==================================
Tool Calls:
  multiply_tool (fc_8fb209ff-321f-443f-a56f-6b99a57dd804)
 Call ID: fc_8fb209ff-321f-443f-a56f-6b99a57dd804
  Args:
    a: 3
    b: 4
================================= Tool Message =================================
Name: multiply_tool

12
================================== Ai Message ==================================
Tool Calls:
  multiply_tool (fc_b435fb1e-9bc8-423a-a863-9156d583acff)
 Call ID: fc_b435fb1e-9bc8-423a-a863-9156d583acff
  Args:
    a: 12
    b: 5
================================= Tool Message =================================
Name: multiply_tool

60
================================== Ai Message ==================================

Model call limits exceeded: run limit (2/2)
